# Chapter 05-05 · Residuals: reading the errors the model leaves

**Label:** Core  |  **Time:** ~50 minutes  |  **Difficulty:** easy to do, and the highest
value-per-minute in this module

**Prerequisites:** 05-04 for the metrics, 05-03 for multiple regression, 02-04 for reading a scatter plot.

**Position in the learning path:** module 05, chapter 5 of 12.

---

## Why this matters

Every metric in 05-04 does the same thing: it takes a list of residuals and crushes it into one number.
That is what makes a metric reportable. It is also what makes it **blind**.

Below are four models. All four have **RMSE 2.4000** - not approximately, exactly, because the data was
calibrated to make them agree. All four have an R-squared between 0.83 and 0.85. Any report that quotes a
metric would describe them as the same model.

**One of them is fine. The other three are each broken in a different, visible, fixable way**, and five
minutes with a plot tells you which is which.

This is the cheapest diagnostic in machine learning, and the one beginners most often skip.

## What you will be able to do

- Compute residuals and plot them the way that works, not the way that looks natural
- Name the four patterns that appear in practice, and what each one means the model is missing
- Turn "the residuals look wrong" into a specific next action
- Recognise heteroscedasticity, and say honestly what transforming the target does and does not fix
- Find, in a real dataset, a group of rows the model is systematically wrong about

## Warm-up: retrieve, do not reread

1. What is the RMSE-to-MAE ratio when every error is the same size, and what does a large value mean?
2. What does a negative R-squared say about the model?
3. In 05-03, what did adding the size column do to the age coefficient, and why?

<br>

*Answers: (1) 1; a large value means a few rows carry most of the error, so go and read them.
(2) It is worse than predicting the mean - negative skill. (3) It flipped from -1.03 to +0.82, because
age was standing in for size until size was present.*

## What a residual is

For each row, the **residual** is what the model got wrong, with its sign kept:

$$e_i = y_i - \hat{y}_i$$

Positive means the model **under-predicted** that row - the truth was higher than the guess. Negative
means it over-predicted.

**The sign is the entire point of this chapter.** Every metric in 05-04 threw the sign away, by taking an
absolute value or a square. That is correct for scoring - a miss is a miss in either direction - and it
is exactly what hides the structure. Four rows that are +5, -5, +5, -5 and four rows that are +5, +5,
+5, +5 have the same MAE and tell completely different stories.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

# SYNTHETIC: four models over the same 200 x-values, deliberately calibrated
# so that all four have identical RMSE. The differences are only in structure.
rng = np.random.default_rng(41)
n_rows = 200
x = rng.uniform(0, 10, n_rows)
noise = rng.normal(0, 1, n_rows)        # one shared draw, scaled per scenario
group = rng.integers(0, 2, n_rows)      # an unrecorded two-category split

scenarios = {
    "A  plain noise": 3 + 2 * x + 2.4875 * noise,
    "B  curvature":   3 + 2 * x + 0.30 * (x - 5) ** 2 + 1.4403 * noise,
    "C  fanning":     3 + 2 * x + noise * (0.15 + 0.3854 * x),
    "D  two groups":  3 + 2 * x + 4.5 * (group - 0.5) + 0.8437 * noise,
}

fits = {}
for name, y in scenarios.items():
    model = LinearRegression().fit(x.reshape(-1, 1), y)
    fitted = model.predict(x.reshape(-1, 1))
    fits[name] = {"y": y, "fitted": fitted, "residual": y - fitted}

summary = pd.DataFrame([
    {"model": name,
     "RMSE": np.sqrt((f["residual"] ** 2).mean()),
     "MAE": np.abs(f["residual"]).mean(),
     "R2": 1 - (f["residual"] ** 2).sum() / ((f["y"] - f["y"].mean()) ** 2).sum()}
    for name, f in fits.items()])
print(summary.to_string(index=False, float_format=lambda v: "%.4f" % v))

**Four different problems, one RMSE.**

The MAE does differ a little - 1.72 to 2.26 - and that is worth noticing, because it is the RMSE-to-MAE
ratio of 05-04 quietly signalling that the *shape* of the errors is not the same. But it tells you
nothing about **what** is different, and a report quoting RMSE alone would not even show that much.

### Predict before running

Before the next cell, commit to answers:

- Which one is the honest model - the one where nothing is wrong?
- For the model with curvature, will its residuals be positive or negative at the **edges** of the x range?
- For the two-group model, what would you see if you could colour the residuals by group?

In [ ]:
thirds = pd.cut(x, [0, 10 / 3, 20 / 3, 10], labels=["low x", "mid x", "high x"])

table = pd.DataFrame({name: pd.Series(f["residual"]).groupby(thirds, observed=True).mean()
                      for name, f in fits.items()})
print("MEAN residual within each third of x")
print(table.round(3).to_string(), "\n")

spread = pd.DataFrame({name: pd.Series(f["residual"]).groupby(thirds, observed=True).std()
                       for name, f in fits.items()})
print("STANDARD DEVIATION of the residuals within each third of x")
print(spread.round(3).to_string(), "\n")

by_group = pd.DataFrame({name: pd.Series(f["residual"]).groupby(group).mean()
                         for name, f in fits.items()})
by_group.index = ["group 0", "group 1"]
print("MEAN residual within each unrecorded group")
print(by_group.round(3).to_string())

**Read the three tables before reading on. Each broken model gives itself away in exactly one of them.**

- **B** is caught by the first table: mean residual **+1.16, -2.25, +0.87** across the thirds. The model is
  too low at both ends and too high in the middle - which is what fitting a straight line through a curve
  always looks like. A mean residual that changes systematically with x is the signature.
- **C** is invisible in the first table (its means are near zero) and obvious in the second: the spread
  goes **0.94, 2.08, 3.34**. The model is right on average everywhere and much less reliable at high x.
- **D** is invisible in both, and unmissable in the third: **-2.21** for group 0, **+2.30** for group 1.
  The model is systematically wrong for everybody, in opposite directions.
- **A** shows nothing anywhere, which is what "nothing is wrong" looks like.

**Notice what just happened.** Three completely different faults, none of which any metric could see, all
found with `groupby` and a mean. You do not need the plots to find these - but the plots find them
faster, and find the ones you did not think to group by.

## The plot, and the version of it that does not work

The standard diagnostic is **residuals against fitted values**: predictions on the x-axis, residuals on
the y-axis, a horizontal line at zero.

It is tempting to plot residuals against the **actual** values instead - that is the number you care
about, after all. Do not. Here is why, and it is not a matter of taste.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(15, 7.2), sharey="row")

for column, (name, f) in enumerate(fits.items()):
    top = axes[0, column]
    top.scatter(f["fitted"], f["residual"], s=16, alpha=0.6, color="#0072B2")
    top.axhline(0, color="#D55E00", linewidth=1.8)
    top.set_title(name, fontsize=11)
    top.set_xlabel("fitted value")
    if column == 0:
        top.set_ylabel("residual\n(vs FITTED - correct)")

    bottom = axes[1, column]
    bottom.scatter(f["y"], f["residual"], s=16, alpha=0.6, color="#999999")
    bottom.axhline(0, color="#D55E00", linewidth=1.8)
    bottom.set_xlabel("actual value")
    if column == 0:
        bottom.set_ylabel("residual\n(vs ACTUAL - misleading)")

plt.tight_layout()
plt.show()

**The top row diagnoses; the bottom row does not.**

Every panel in the bottom row slopes upward, including panel A where nothing is wrong. That upward slope
is not a finding about the data. It is arithmetic:

$$y = \hat{y} + e \quad \Rightarrow \quad \text{cov}(e, y) = \text{var}(e)$$

so the residuals are **guaranteed** to correlate positively with the actuals, in every least-squares fit
ever performed. The correlation even has a closed form.

In [ ]:
checks = []
for name, f in fits.items():
    r_squared = 1 - (f["residual"] ** 2).sum() / ((f["y"] - f["y"].mean()) ** 2).sum()
    checks.append({"model": name,
                   "corr(residual, fitted)": np.corrcoef(f["residual"], f["fitted"])[0, 1],
                   "corr(residual, actual)": np.corrcoef(f["residual"], f["y"])[0, 1],
                   "sqrt(1 - R2)": np.sqrt(1 - r_squared)})
print(pd.DataFrame(checks).round(4).to_string(index=False))

Two exact results, both worth memorising:

> **corr(residual, fitted) = 0, always.** Least squares makes it so - if any of the fitted values' signal
> were left in the residuals, the fit would not have been optimal. So the fitted-value plot has a known,
> flat null hypothesis: *any* structure you see in it is real.
>
> **corr(residual, actual) = sqrt(1 - R²), always.** A model with R-squared 0.83 will show a correlation
> of 0.41 between its residuals and the truth, on perfect data, with nothing wrong.

That second line is the trap. The slope in the bottom row is a restatement of the R-squared, and a
beginner who plots it sees "my errors are bigger when the value is bigger" and starts fixing a problem
that does not exist. **A better model makes that plot look *worse*, not better** - as R-squared rises,
sqrt(1 - R²) falls, so the spurious correlation shrinks. The diagnostic runs backwards.

**The rule: residuals against fitted values.** The reference line is flat, and it is flat because of a
theorem rather than because of luck.

## The four patterns, and what to do about each

Now read the top row again. Each panel is one of the four things you will actually meet.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 3.9))
titles = ["random cloud\nnothing to do",
          "a curve\nthe model is missing a shape",
          "a fan\nthe spread depends on the prediction",
          "two bands\na group the model cannot see"]
colours = ["#009E73", "#D55E00", "#0072B2", "#7B3294"]

for ax, (name, f), title, colour in zip(axes, fits.items(), titles, colours):
    if name.startswith("D"):
        for value, marker, shade in [(0, "o", "#7B3294"), (1, "^", "#E69F00")]:
            keep = group == value
            ax.scatter(f["fitted"][keep], f["residual"][keep], s=16, alpha=0.7,
                       marker=marker, color=shade, label="group %d" % value)
        ax.legend(fontsize=8, loc="upper right")
    else:
        ax.scatter(f["fitted"], f["residual"], s=16, alpha=0.6, color=colour)
    ax.axhline(0, color="#000000", linewidth=1.5)
    ax.set_title(title, fontsize=10.5)
    ax.set_xlabel("fitted value")
axes[0].set_ylabel("residual")

plt.tight_layout()
plt.show()

| Pattern | What it means | What to do |
|---|---|---|
| **Random cloud** around zero | The model has extracted the structure it can | Nothing. Stop looking |
| **A curve** (∪ or ∩) | A relationship is not straight | Add a squared term or a transform (05-07), or use a model that bends (05-10) |
| **A fan** (spread grows) | The uncertainty depends on the prediction | Consider a target transform, and *report* the varying uncertainty rather than one RMSE |
| **Bands or clumps** | A variable the model does not have | Find the variable and add it. This is the most valuable finding of the four |

**The last one is the one to hope for.** Curvature and fanning are properties of the problem. A band is a
missing column, and a missing column is something you can go and get.

A fifth pattern deserves a mention because it is a bug rather than a finding: **residuals that trend with
row order**. If plotting residuals against the row index shows drift, the rows are not exchangeable -
usually because the file is sorted by date, and 04-04 already covered what that means for splitting.

## Locating the fault: residuals against each feature

The fitted-value plot tells you *that* something is wrong. With more than one feature it does not tell
you **which column** is responsible. For that, plot the residuals against each feature in turn.

In [ ]:
# SYNTHETIC: 400 deliveries. TRUTH: minutes rises linearly with distance,
# and QUADRATICALLY with the number of stops - which the model is not told.
delivery_rng = np.random.default_rng(6)
n_deliveries = 400
distance_km = delivery_rng.uniform(1, 20, n_deliveries)
stops = delivery_rng.uniform(1, 12, n_deliveries)
minutes = (8 + 1.9 * distance_km + 0.55 * stops ** 2
           + delivery_rng.normal(0, 3.0, n_deliveries))

features = pd.DataFrame({"distance_km": distance_km, "stops": stops})
route_model = LinearRegression().fit(features, minutes)
route_residual = minutes - route_model.predict(features)

print("RMSE %.3f minutes   R2 %.4f" % (np.sqrt((route_residual ** 2).mean()),
                                       route_model.score(features, minutes)))
print("\nmean residual by third:")
for column in features.columns:
    parts = pd.qcut(features[column], 3, labels=["low", "mid", "high"])
    means = pd.Series(route_residual).groupby(parts, observed=True).mean().round(3)
    print("  %-12s %s" % (column, means.to_dict()))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14.5, 4.1))

axes[0].scatter(route_model.predict(features), route_residual, s=14, alpha=0.55, color="#666666")
axes[0].set_xlabel("fitted value")
axes[0].set_title("Against fitted: something is wrong", fontsize=11)

for ax, column, colour in [(axes[1], "distance_km", "#009E73"), (axes[2], "stops", "#D55E00")]:
    ax.scatter(features[column], route_residual, s=14, alpha=0.55, color=colour)
    ax.set_xlabel(column)
    ax.set_title("Against %s" % column, fontsize=11)

for ax in axes:
    ax.axhline(0, color="#000000", linewidth=1.5)
axes[0].set_ylabel("residual (minutes)")

plt.tight_layout()
plt.show()

**`distance_km` is clean. `stops` holds the whole curve.** The middle panel is the random cloud you want;
the right panel is an unmistakable ∪.

The numbers say the same thing without the plot: against `stops`, the mean residual by third runs
**+2.36, -4.69, +2.31**, while against `distance_km` it barely moves.

**That is a specific instruction, not a vague worry.** The model is missing a shape in `stops`, so give
it one.

In [ ]:
extended = features.assign(stops_squared=features["stops"] ** 2)
fixed_model = LinearRegression().fit(extended, minutes)
fixed_residual = minutes - fixed_model.predict(extended)

print("before adding stops_squared:  RMSE %.4f   R2 %.4f"
      % (np.sqrt((route_residual ** 2).mean()), route_model.score(features, minutes)))
print("after:                        RMSE %.4f   R2 %.4f"
      % (np.sqrt((fixed_residual ** 2).mean()), fixed_model.score(extended, minutes)))
print("\nthe noise the data was built with had sd 3.0")
print("\nmean residual against stops, by third, after the fix:")
parts = pd.qcut(features["stops"], 3, labels=["low", "mid", "high"])
print((pd.Series(fixed_residual).groupby(parts, observed=True).mean().round(3) + 0.0).to_dict())

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(12.5, 4.2), sharey=True)

left.scatter(features["stops"], route_residual, s=14, alpha=0.55, color="#D55E00")
left.set_title("Before: the residuals remember what the model forgot", fontsize=11)
right.scatter(features["stops"], fixed_residual, s=14, alpha=0.55, color="#009E73")
right.set_title("After adding stops_squared: nothing left to see", fontsize=11)

for ax in (left, right):
    ax.axhline(0, color="#000000", linewidth=1.5)
    ax.set_xlabel("stops")
left.set_ylabel("residual (minutes)")

plt.tight_layout()
plt.show()

**RMSE falls from 5.56 to 3.00 minutes, and the residuals go flat.**

The 3.00 is the number to dwell on. The data was built with noise of standard deviation **3.0**, so the
fixed model is now wrong by exactly as much as the data is unpredictable, and not one minute more.

Notice also what the R-squared was doing while the model was broken: **0.9500**. A number that would be
celebrated in most reports, on a model with a visible, fixable fault costing it nearly half its error. The
residual plot has gone flat because **there is nothing left in it** - which is what a finished model
looks like.

This is the whole loop, and it is worth stating as a habit:

> **Fit → plot the residuals against every feature → if a shape appears, that feature needs a term →
> add it → plot again.** Stop when the plots are boring.

## Heteroscedasticity, and the honest version of the standard advice

**Heteroscedasticity** is the long word for the fan: the spread of the residuals depends on the
prediction. It is extremely common, because many quantities are naturally **multiplicative** - a house
worth 500,000 is uncertain by tens of thousands, a house worth 50,000 is uncertain by thousands, and
neither is uncertain by "the same amount".

The standard advice is "take logs". That advice is right about one thing and quietly wrong about another,
so it is worth doing carefully.

In [ ]:
# SYNTHETIC: 300 houses where the error is MULTIPLICATIVE by construction.
# TRUTH: price = 2.5 x size, times a random factor averaging 1.
house_rng = np.random.default_rng(19)
n_h = 300
house_size = house_rng.uniform(40, 240, n_h)
house_price = 2.5 * house_size * np.exp(house_rng.normal(0, 0.22, n_h))

size_column = house_size.reshape(-1, 1)
raw_fit = LinearRegression().fit(size_column, house_price)
raw_residual = house_price - raw_fit.predict(size_column)

log_fit = LinearRegression().fit(size_column, np.log(house_price))
log_residual = np.log(house_price) - log_fit.predict(size_column)
back_prediction = np.exp(log_fit.predict(size_column))
back_residual = house_price - back_prediction


def spread_by_third(prediction, residual, label):
    parts = pd.qcut(prediction, 3, labels=["low", "mid", "high"])
    sds = pd.Series(residual).groupby(parts, observed=True).std()
    print("%-34s %s   high/low ratio %.2f"
          % (label, np.round(sds.to_numpy(), 3), sds.iloc[2] / sds.iloc[0]))


spread_by_third(raw_fit.predict(size_column), raw_residual,
                "residual sd, price scale, raw fit")
spread_by_third(log_fit.predict(size_column), log_residual,
                "residual sd, LOG scale, log fit")
spread_by_third(back_prediction, back_residual,
                "residual sd, price scale, log fit")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.1))

axes[0].scatter(raw_fit.predict(size_column), raw_residual, s=15, alpha=0.6, color="#D55E00")
axes[0].set_title("Fit on price: a clear fan (3.33x)", fontsize=11)
axes[0].set_xlabel("fitted price")
axes[0].set_ylabel("residual (price units)")

axes[1].scatter(log_fit.predict(size_column), log_residual, s=15, alpha=0.6, color="#009E73")
axes[1].set_title("Fit on log price, read on the LOG scale (1.06x)", fontsize=11)
axes[1].set_xlabel("fitted log price")
axes[1].set_ylabel("residual (log units)")

axes[2].scatter(back_prediction, back_residual, s=15, alpha=0.6, color="#0072B2")
axes[2].set_title("The same fit, read back on the PRICE scale (3.38x)", fontsize=11)
axes[2].set_xlabel("fitted price, exponentiated")
axes[2].set_ylabel("residual (price units)")

for ax in axes:
    ax.axhline(0, color="#000000", linewidth=1.5)

plt.tight_layout()
plt.show()

**The middle panel is flat. The right-hand panel fans out just like the left one did.**

Both of those facts are true at the same time, and the honest summary is:

> **Taking logs fixed the diagnostic, not the uncertainty.** On the log scale the model's error really is
> constant - the residual spread goes from a 3.33x ratio to **1.06x** - which is strong evidence that the
> process is genuinely multiplicative, and that is a real finding about the data. But an expensive house
> is still harder to price in euros than a cheap one, and no transform can change that, because it is a
> fact about houses rather than about the model.

There is a second cost that is almost always left out.

In [ ]:
comparison = pd.DataFrame([
    {"fitted on": "price", "RMSE (price units)": np.sqrt((raw_residual ** 2).mean()),
     "MAE (price units)": np.abs(raw_residual).mean(),
     "mean residual, mid third": raw_residual[pd.qcut(raw_fit.predict(size_column), 3,
                                                      labels=False) == 1].mean()},
    {"fitted on": "log price", "RMSE (price units)": np.sqrt((back_residual ** 2).mean()),
     "MAE (price units)": np.abs(back_residual).mean(),
     "mean residual, mid third": back_residual[pd.qcut(back_prediction, 3,
                                                       labels=False) == 1].mean()},
])
print(comparison.round(3).to_string(index=False))

**The log model is worse in price units - RMSE 100.7 against 94.4 - and it is biased.**

That is not a mistake in the code. `exp` of the mean of the logs is the *geometric* mean, which is
always below the arithmetic mean, so exponentiating a log-scale prediction systematically under-predicts
the price. The mid-third mean residual of **+31.8** is that bias, in euros, visible in the right-hand
panel as a cloud sitting above the zero line in the middle.

So the decision is a real trade-off, not a free improvement:

- **Fit on the log** when you care about *relative* error, when the diagnostics must be readable, or when
  the multiplicative story is the true one. Expect to correct the back-transformation if you report in
  the original units.
- **Fit on the raw target** when the report is in the original units and squared error in those units is
  what the business pays for.

**And in both cases, report the spread, not just the RMSE.** "Typically 94 out, but 41 out on cheap
houses and 138 out on expensive ones" is three numbers, all of them true, and far more useful than one.

## The residual histogram, and why it matters least

The fourth standard plot is a histogram of the residuals, checked for a bell shape. It is the one
beginners are most often told to make and the one that matters least, so it is worth being precise about
what it is for.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 3.6), sharey=True)

for ax, (name, f), colour in zip(axes, fits.items(),
                                 ["#009E73", "#D55E00", "#0072B2", "#7B3294"]):
    ax.hist(f["residual"], bins=25, color=colour, alpha=0.8)
    ax.axvline(0, color="#000000", linewidth=1.5)
    ax.set_title("%s\nskew %+.2f   excess kurtosis %+.2f"
                 % (name, pd.Series(f["residual"]).skew(), pd.Series(f["residual"]).kurt()),
                 fontsize=10)
    ax.set_xlabel("residual")
axes[0].set_ylabel("rows")

plt.tight_layout()
plt.show()

**It catches one of the three faults, half-catches a second, and misses the third completely.**

- **D is caught, and loudly.** Two humps, excess kurtosis **-1.60**: a flat-topped shape that is a
  histogram's way of saying "these are two populations". This is the fault the histogram is genuinely
  good at, because a group offset shifts residuals without reference to the prediction.
- **C is half-caught.** Skew is only +0.04, but excess kurtosis is **+3.30** - long tails in both
  directions. That tells you a few rows are far out; it does not tell you they are the high-prediction
  rows, which is the actionable half.
- **B is missed entirely.** Skew +0.25, excess kurtosis -0.36 - as respectable as A's +0.22 and +0.09.
  Curvature bends the residuals *with the prediction*, and the histogram is exactly the plot that throws
  the prediction away.

That is the rule in one line: **the histogram sees the residuals' distribution, and the fitted-value plot
sees their relationship to the model.** Most faults live in the relationship.

**What it is genuinely good for:**

- **Spotting a long tail** - the RMSE-to-MAE ratio of 05-04 in picture form, as C shows.
- **Spotting a bimodal shape** - two humps mean two populations, as D shows.
- **Spotting a hard edge** - a histogram that stops dead at a value, which usually means the target is
  capped. The next section is exactly that.

**What it is not for:** verifying that the residuals are normally distributed. Least squares does not
require normal residuals to give good predictions; that assumption belongs to certain *confidence
intervals*, which is a different task from the one in this module. Skipping a normality test costs you
almost nothing. Skipping the fitted-value plot costs you the three faults above.

## Failure lab: a residual plot on real data

Everything so far was synthetic, so the fault was always findable by construction. Here is the same
procedure on a real dataset, where nobody planted anything.

**Dataset:** California Housing.
**Source:** `sklearn.datasets.fetch_california_housing`, derived from the 1990 US Census; the original
paper is Pace & Barry (1997), *Sparse Spatial Autoregressions*.
**Unit of observation:** one census block group - a few hundred to a few thousand residents, **not** one
house.
**Target:** `MedHouseVal`, the median house value in that block group, in hundreds of thousands of dollars.
**Time meaning:** a snapshot of 1990. It says nothing about house prices today.
**Retrieval:** downloaded by scikit-learn on first use and cached locally.

In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

california = fetch_california_housing(as_frame=True)
X_all, y_all = california.data, california.target

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.25, random_state=0)

census_model = LinearRegression().fit(X_train, y_train)
test_prediction = census_model.predict(X_test)
test_residual = y_test.to_numpy() - test_prediction

print("rows: %d train, %d test" % (len(X_train), len(X_test)))
print("test RMSE %.4f   MAE %.4f   R2 %.4f"
      % (np.sqrt((test_residual ** 2).mean()), np.abs(test_residual).mean(),
         census_model.score(X_test, y_test)))

R-squared 0.59 on held-out rows: an ordinary result, neither embarrassing nor good. A report would stop
here.

### Predict before running

The residual plot is next. Before you look: is there anything in the *description above* that should make
you expect a specific shape? Read the target line again.

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(13, 4.6))

at_cap = y_test.to_numpy() >= 5.00001
left.scatter(test_prediction[~at_cap], test_residual[~at_cap], s=8, alpha=0.28,
             color="#0072B2", label="ordinary rows")
left.scatter(test_prediction[at_cap], test_residual[at_cap], s=10, alpha=0.6,
             color="#D55E00", label="target at its maximum, 5.00001")
left.axhline(0, color="#000000", linewidth=1.5)
left.set_xlabel("predicted median value ($100k)")
left.set_ylabel("residual")
left.set_title("A hard diagonal edge, not a cloud", fontsize=11)
left.legend(fontsize=8.5, loc="upper right")

right.hist(y_all, bins=60, color="#666666")
right.axvline(5.00001, color="#D55E00", linewidth=2,
              label="%d rows pile up here (%.2f%%)"
                    % (int((y_all >= 5.00001).sum()), 100 * float((y_all >= 5.00001).mean())))
right.set_xlabel("median house value ($100k)")
right.set_ylabel("block groups")
right.set_title("The target itself, and why", fontsize=11)
right.legend(fontsize=8.5)

plt.tight_layout()
plt.show()

**That straight diagonal line of points is the fault, and it is in the data, not the model.**

The target is **censored**: every block group worth more than $500,000 was recorded as exactly 5.00001,
because that was the top code in the source. **965 rows out of 20,640 - 4.68% - sit on that ceiling.**

The diagonal is the arithmetic of a ceiling. For those rows the actual is fixed at 5.00001, so
`residual = 5.00001 - prediction`, which is a straight line of slope -1 against the prediction. Any
plot of residuals against fitted values will show it, on any model, forever.

Here is what it costs.

In [ ]:
damage = pd.DataFrame([
    {"rows": "at the $500k ceiling", "n": int(at_cap.sum()),
     "mean residual": test_residual[at_cap].mean(), "MAE": np.abs(test_residual[at_cap]).mean()},
    {"rows": "everything else", "n": int((~at_cap).sum()),
     "mean residual": test_residual[~at_cap].mean(), "MAE": np.abs(test_residual[~at_cap]).mean()},
])
print(damage.round(4).to_string(index=False))
print("\nMAE is %.2f times worse on the censored rows"
      % (np.abs(test_residual[at_cap]).mean() / np.abs(test_residual[~at_cap]).mean()))

**MAE 1.4937 on the censored rows against 0.4914 on the rest - three times worse - and the mean residual
there is +1.0749**, meaning the model under-predicts every one of them by more than $100,000 on average.

**Except that it does not.** The model is very likely *right* about those block groups and the recorded
value is wrong: a neighbourhood the model prices at $650,000 is stored as $500,000 because the census
would not record more. The residual is measuring the ceiling, not the model.

**Which is exactly why this belongs in a residuals chapter.** No metric on the test set could have told
you this. RMSE 0.7351 is a single number that silently averages a real error over 95% of the rows with a
recording artefact over the other 5%. One plot separated them in about three seconds.

**What to do about it** is a genuine decision, not a recipe:

- **Report separately.** Quote the MAE on the uncensored rows as the model's accuracy, and state the
  censoring as a known limitation. Honest, and usually right.
- **Exclude the capped rows from training** so the model is not taught that expensive neighbourhoods are
  worth 5.0. This helps the fit and makes the test metric less comparable to published numbers.
- **Model it properly** as a censored target, which is a real technique and out of scope here.

What you must not do is quote 0.7351 without mentioning it.

## The whole procedure on one page

In [ ]:
fig, ax = plt.subplots(figsize=(12.5, 5.4))
ax.set_xlim(0, 10)
ax.set_ylim(0, 7.1)
ax.axis("off")

steps = [
    (0.3, 5.4, "#DDEBF7", "1. Fit the model", "any model, any metric"),
    (0.3, 4.2, "#DDEBF7", "2. residual = actual - prediction", "keep the sign"),
    (0.3, 3.0, "#DDEBF7", "3. Plot residual vs FITTED", "never vs actual"),
    (0.3, 1.8, "#DDEBF7", "4. Plot residual vs each feature", "this locates the column"),
    (0.3, 0.6, "#DDEBF7", "5. Plot residual vs row order", "catches sorted data"),
]
for x0, y0, colour, title, note in steps:
    ax.add_patch(plt.Rectangle((x0, y0), 4.2, 0.95, facecolor=colour,
                               edgecolor="#0072B2", linewidth=1.4))
    ax.text(x0 + 0.18, y0 + 0.58, title, fontsize=11, fontweight="bold")
    ax.text(x0 + 0.18, y0 + 0.22, note, fontsize=9, color="#444444")

verdicts = [
    (5.2, 5.4, "#D9EAD3", "random cloud", "done - stop looking"),
    (5.2, 4.2, "#FCE5CD", "a curve", "add a term or bend the model"),
    (5.2, 3.0, "#FCE5CD", "a fan", "transform, or report the spread"),
    (5.2, 1.8, "#F4CCCC", "bands or clumps", "find the missing column"),
    (5.2, 0.6, "#F4CCCC", "a hard straight edge", "the target is capped"),
]
for x0, y0, colour, title, note in verdicts:
    ax.add_patch(plt.Rectangle((x0, y0), 4.5, 0.95, facecolor=colour,
                               edgecolor="#666666", linewidth=1.2))
    ax.text(x0 + 0.18, y0 + 0.58, title, fontsize=11, fontweight="bold")
    ax.text(x0 + 0.18, y0 + 0.22, note, fontsize=9, color="#444444")

ax.annotate("", xy=(5.1, 3.5), xytext=(4.6, 3.5),
            arrowprops=dict(arrowstyle="-|>", linewidth=2.2, color="#0072B2"))
ax.text(4.85, 3.65, "you see", fontsize=9, ha="center", color="#0072B2")
ax.text(2.4, 6.65, "WHAT YOU DO", fontsize=11.5, fontweight="bold", ha="center")
ax.text(7.45, 6.65, "WHAT IT MEANS", fontsize=11.5, fontweight="bold", ha="center")

plt.tight_layout()
plt.show()

## Common misconceptions

**"The residuals should be normally distributed, so I ran a normality test."**
Least squares does not need normal residuals to predict well, and the histogram missed the curvature in
this chapter's panel B entirely. Run the fitted-value plot instead. Normality matters for certain
confidence intervals, which is a different job.

**"Residuals against the actual values - same thing, surely."**
No. That plot slopes upward for every least-squares fit ever made, with correlation exactly
`sqrt(1 - R²)`, and it slopes *more* for worse models. It is a picture of the R-squared, not a diagnostic.

**"My residual plot is fine, so the model is right."**
It means nothing is wrong in the directions you plotted. A group split you did not think to colour by is
invisible until you colour by it - which is why the feature-by-feature sweep matters, and why the
question "what column do I not have?" never fully closes.

**"Taking logs fixed the heteroscedasticity."**
It fixed it on the log scale. In the original units the spread came back at a ratio of 3.38, and the
back-transformed predictions picked up a bias of about +32. Both facts are true; report both.

**"The model is badly wrong on those rows, so I should improve the model."**
Sometimes the rows are wrong, not the model. The California ceiling is a recording rule, and a "better"
model would only learn to reproduce a censoring artefact. Look at the rows before changing the model.

**"Residual plots are for linear models."**
Every model that produces a number produces residuals. The plots in this chapter work unchanged on the
random forests of 05-10 and the gradient boosting of 05-11, and are just as informative there.

## Exercises

Solutions: `solutions/05_regression/05-05_residuals_solutions.ipynb`.

### Quick understanding

**E1.** Define the residual, and say why keeping its sign is the entire point of this chapter.

**E2.** Why is `corr(residual, fitted)` exactly zero for a least-squares fit? Answer in one sentence about
optimality, not about algebra.

**E3.** Name the four patterns and give the action each one implies, in one clause each.

### Hand calculation

**E4.** Actuals 10, 12, 15, 19, 24 and predictions 11, 13, 15, 17, 19. Write the five residuals, then say
what pattern they show and what it means.

**E5.** For the same five rows, compute the mean residual over the first two rows and over the last two.
Explain why those two numbers are a curvature test.

**E6.** A model has R-squared 0.64. Without running anything, give the correlation you expect between its
residuals and the actual values.

**E7.** Residuals for six rows are +4, -4, +4, -4, +4, -4, and for six other rows are +4, +4, +4, -4, -4,
-4. Give MAE and RMSE for each set, then say which set you would rather have and why the metrics cannot
help you decide.

### Coding

**E8.** Write `diagnose(model, X, y)` that prints the mean and standard deviation of the residuals within
each third of every feature, and flags any feature where the means differ by more than one residual
standard deviation. Run it on this chapter's route data before and after the fix.

**E9.** Reproduce the identity `corr(residual, actual) = sqrt(1 - R²)` on three fits with very different
R-squared values, and show it holds to at least four decimal places.

**E10.** Take scenario D and give the model the group column. Report what happens to RMSE, and check the
residual plot again.

**E11.** Sort this chapter's route data by `stops`, refit, and plot residuals against **row index**.
Explain what you see, and then do the same on the *unsorted* data. Which of the two plots is telling you
about the model?

**E12.** Fit the California model, then plot residuals against `Latitude` and `Longitude`. Report what you
find and say what feature you would want.

### Interpretation

**E13.** A colleague's residual plot is a perfect flat band, but their RMSE-to-MAE ratio is 4.1 on 300
rows. Reconcile the two observations.

**E14.** Your residual plot shows a fan, and the business reports error in the target's own units. State
what you would report, in one sentence a manager could repeat.

### Debugging

**E15.** A residual plot shows a tight diagonal line of points, sloping downward, containing about 8% of
the rows. Give the two most likely causes and how to tell them apart in one line of code.

**E16.** After adding a squared term the residual plot went flat, but the held-out RMSE got worse. What
happened, and what would you check?

### Exam and interview reasoning

**E17.** "How do you know when a regression model is finished?" Answer in under a minute, then handle the
follow-up: "what if the residual plot is flat but the error is still too large for the business?"

### Transfer to a different situation

**E18.** You are predicting hospital length of stay in days. The residual plot shows a fan *and* a hard
edge at the low end. Explain both, and say which one is a modelling problem and which is a data property.

### Explain it to someone non-technical

**E19.** In under 90 words, explain to a manager why you want to spend an afternoon looking at the model's
mistakes when the accuracy number is already acceptable.

### Optional challenge

**E20.** Construct two datasets on the same x-values whose linear fits have **identical** RMSE, MAE **and**
R-squared, but whose residual plots show opposite faults. Then state what this proves about reporting
metrics without plots.

**E21.** Show empirically that `corr(residual, fitted) = 0` holds only when the model includes an
intercept, by refitting without one and measuring the correlation.

## Mastery check

- [ ] Compute residuals and plot them against fitted values without looking anything up
- [ ] Explain, from the algebra, why plotting against actuals is wrong
- [ ] Name what each of the four patterns means and what to do about it
- [ ] Find the responsible feature when a fault appears, rather than guessing
- [ ] Say what a log transform of the target does and does not fix
- [ ] Recognise a censored target from the shape of its residuals

## What should now feel instinctive

- Plotting the residuals before quoting the metric, every time
- Treating a shape in a residual plot as an instruction rather than a worry
- Grouping the residuals by anything you have, especially columns not in the model
- Asking "wrong model, or wrong rows?" before changing the model
- Reporting the spread of the error alongside its average

## Flashcards

| Front | Back |
|---|---|
| Residual | `actual - prediction`, with the sign kept |
| Residuals vs fitted | The diagnostic. Reference line is flat by a theorem |
| Residuals vs actual | Always slopes up. corr = `sqrt(1 - R²)`. Not a diagnostic |
| corr(residual, fitted) | Exactly 0 for any least-squares fit with an intercept |
| Random cloud | Nothing left to extract. Stop |
| A ∪ or ∩ shape | A relationship is not straight - add a term or bend the model |
| A fan | Heteroscedasticity - the uncertainty depends on the prediction |
| Bands or clumps | A missing column. The most valuable finding |
| A hard diagonal edge | The target is capped or censored |
| Four models, one RMSE | 2.4000 each; three of them broken in different ways |
| Log transform | Fixes the diagnostic on the log scale; spread returned at 3.38x in price units |
| California ceiling | 4.68% of rows at 5.00001; MAE 3x worse there, and the data is what is wrong |

## Next

**05-06 · Gradient descent from scratch.** Up to now every fit has come from `LinearRegression()`, which
solves the problem in closed form - and nothing outside linear regression can do that. The next chapter
throws away the closed form and finds the same coefficients by walking downhill, using the gradients of
03-08 on the loss of 05-02.

That is the machinery every model in the rest of this course is trained with, and once the loop is in
your hands the same twenty lines will train a neural network in module 10.